# Cheap feature labels, scored

Claude Haiku 5.5 labels 50 Gemma-2-2B transcoder features from their Neuronpedia examples. Every label, ours and Neuronpedia's, is scored by the same detector on the same held-out records. The replies are cached in `data/labels_llm_cache.jsonl`, so this runs without an API key.

In [1]:
import gzip
import json
from pathlib import Path

from circuitcheck.labels import AnthropicLLM, CachedLLM, FeatureData, label_feature

payload = json.loads(gzip.decompress(Path("data/labels_gemma_features.json.gz").read_bytes()))
feats = [FeatureData.model_validate(x) for x in payload["features"]]
llm = CachedLLM(AnthropicLLM("claude-haiku-5-5"), "data/labels_llm_cache.jsonl")
lab = label_feature(llm, feats[0], score="both")
print(lab.unit, "->", lab.label)
print("detection AUROC", lab.detection.auroc, "| simulation r", round(lab.simulation.correlation, 2))
print("API calls made:", llm.usage.calls, "| cached:", llm.usage.cached_calls)

20:15589 -> Texas place names and legal citations (Texas, San Antonio, Tex. R. Civ. P.), mostly after 'in' or 'Texas'
detection AUROC 0.8984375 | simulation r 0.56
API calls made: 0 | cached: 18


The full run (`examples/scripts/label_gemma_demo.py`):

In [2]:
import statistics as st

r = json.loads(Path("data/labels_gemma_results.json").read_text())


def mean(xs, key):
    v = [
        x[key]["auroc" if key == "detection" else "correlation"]
        for x in xs
        if x[key] and x[key].get("auroc" if key == "detection" else "correlation") is not None
    ]
    return round(st.mean(v), 3)


for k in ("ours", "neuronpedia_long", "neuronpedia_short"):
    print(f"{k:18} detection {mean(r[k], 'detection')}  simulation {mean(r[k], 'simulation')}")
print("calibration", {k: r["calibration"][k] for k in ("confidence_vs_score", "detector_ece")})

ours               detection 0.797  simulation 0.452
neuronpedia_long   detection 0.784  simulation 0.366
neuronpedia_short  detection 0.803  simulation 0.381
calibration {'confidence_vs_score': 0.10711343398893718, 'detector_ece': 0.2581434058898848}


In [3]:
from circuitcheck.labels import FeatureLabel
from circuitcheck.labels.autointerp import Calibration
from circuitcheck.labels.neuronpedia import Comparison
from circuitcheck.report import render_labels, save_html

ours = [FeatureLabel.model_validate(x) for x in r["ours"]]
comps = [Comparison.model_validate(c) for c in r["compare_long"]]
save_html(render_labels(ours, Calibration.model_validate(r["calibration"]), comps), "reports/labels_gemma.html")

PosixPath('reports/labels_gemma.html')